# CAPSTONE PROJECT: Customer Churn Prediction

## Classification Problem

---

## Project Overview

**Objective:** Build a machine learning model to predict whether a customer will churn (leave) a service.

**Business Context:** 
- Customer churn costs businesses millions in lost revenue
- Acquiring new customers is 5-25x more expensive than retaining existing ones
- Early prediction allows proactive retention strategies

**Dataset:** Telecom Customer Churn (Kaggle) or similar churn dataset

---

## Project Structure

```
1. Problem Definition
2. Data Collection
3. Exploratory Data Analysis (EDA)
4. Data Preprocessing
5. Handling Imbalanced Data
6. Model Building
7. Model Evaluation
8. Hyperparameter Tuning
9. Business Insights
10. Conclusions & Recommendations
```

---

## Documentation: Classification Capstone Guide

### Key Considerations for Classification

#### 1. Class Imbalance
- Churn datasets are often imbalanced (more non-churners)
- Techniques: SMOTE, undersampling, class weights
- Use appropriate metrics (not just accuracy!)

#### 2. Metrics Selection
- **Accuracy**: Overall correctness (misleading for imbalanced data)
- **Precision**: Of predicted churners, how many actually churned?
- **Recall**: Of actual churners, how many did we catch?
- **F1 Score**: Balance of precision and recall
- **AUC-ROC**: Overall model quality

#### 3. Business Impact
- False Negative (missed churner) = Lost customer
- False Positive (wrong prediction) = Wasted retention effort
- Usually: Recall > Precision for churn prediction

---

## Part 1: Setup and Data Collection

In [ ]:
# Install required packages
# !pip install numpy pandas matplotlib seaborn scikit-learn imbalanced-learn

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# Metrics
from sklearn.metrics import (accuracy_score, precision_score, recall_score, 
                             f1_score, roc_auc_score, confusion_matrix,
                             classification_report, roc_curve, precision_recall_curve)

# Settings
import warnings
warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 50)

print("Libraries imported successfully!")

In [ ]:
# Create realistic telecom churn dataset
# (In practice, download from Kaggle: https://www.kaggle.com/datasets/blastchar/telco-customer-churn)

np.random.seed(42)
n_samples = 7000

# Generate customer data
def generate_churn_dataset(n_samples):
    np.random.seed(42)
    
    # Customer demographics
    gender = np.random.choice(['Male', 'Female'], n_samples)
    senior_citizen = np.random.choice([0, 1], n_samples, p=[0.84, 0.16])
    partner = np.random.choice(['Yes', 'No'], n_samples)
    dependents = np.random.choice(['Yes', 'No'], n_samples, p=[0.3, 0.7])
    
    # Account information
    tenure = np.random.exponential(25, n_samples).astype(int)
    tenure = np.clip(tenure, 1, 72)
    
    # Services
    phone_service = np.random.choice(['Yes', 'No'], n_samples, p=[0.9, 0.1])
    internet_service = np.random.choice(['DSL', 'Fiber optic', 'No'], n_samples, p=[0.35, 0.45, 0.2])
    contract = np.random.choice(['Month-to-month', 'One year', 'Two year'], n_samples, p=[0.55, 0.21, 0.24])
    paperless_billing = np.random.choice(['Yes', 'No'], n_samples)
    payment_method = np.random.choice(
        ['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'],
        n_samples, p=[0.33, 0.22, 0.22, 0.23]
    )
    
    # Monthly charges (based on services)
    monthly_charges = 20 + np.random.normal(50, 20, n_samples)
    monthly_charges = np.clip(monthly_charges, 18, 120)
    
    # Total charges based on tenure
    total_charges = monthly_charges * tenure + np.random.normal(0, 100, n_samples)
    total_charges = np.clip(total_charges, 18, 10000)
    
    # Generate churn based on realistic factors
    churn_prob = np.zeros(n_samples)
    
    # Month-to-month contracts more likely to churn
    churn_prob += (contract == 'Month-to-month') * 0.25
    
    # Short tenure more likely to churn
    churn_prob += (tenure < 12) * 0.15
    
    # High monthly charges
    churn_prob += (monthly_charges > 70) * 0.1
    
    # Electronic check payment
    churn_prob += (payment_method == 'Electronic check') * 0.1
    
    # Fiber optic (higher churn due to competition)
    churn_prob += (internet_service == 'Fiber optic') * 0.1
    
    # Senior citizens
    churn_prob += senior_citizen * 0.1
    
    # Base probability and noise
    churn_prob += np.random.uniform(0, 0.2, n_samples)
    
    # Convert to binary
    churn = (churn_prob > 0.5).astype(int)
    
    # Create DataFrame
    df = pd.DataFrame({
        'customerID': [f'CUST_{i:05d}' for i in range(n_samples)],
        'gender': gender,
        'SeniorCitizen': senior_citizen,
        'Partner': partner,
        'Dependents': dependents,
        'tenure': tenure,
        'PhoneService': phone_service,
        'InternetService': internet_service,
        'Contract': contract,
        'PaperlessBilling': paperless_billing,
        'PaymentMethod': payment_method,
        'MonthlyCharges': np.round(monthly_charges, 2),
        'TotalCharges': np.round(total_charges, 2),
        'Churn': churn
    })
    
    return df

# Generate dataset
df = generate_churn_dataset(n_samples)

print(f"Dataset Shape: {df.shape}")
df.head()

## Part 2: Exploratory Data Analysis (EDA)

In [ ]:
# Basic info
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"\nShape: {df.shape[0]} customers, {df.shape[1]} columns")
print(f"\nData Types:")
print(df.dtypes)

In [ ]:
# Target variable distribution
print("\n" + "=" * 40)
print("TARGET VARIABLE: CHURN")
print("=" * 40)
churn_dist = df['Churn'].value_counts()
churn_pct = df['Churn'].value_counts(normalize=True) * 100

print(f"\nChurn Distribution:")
print(f"No Churn (0): {churn_dist[0]} ({churn_pct[0]:.1f}%)")
print(f"Churn (1): {churn_dist[1]} ({churn_pct[1]:.1f}%)")

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#4ECDC4', '#FF6B6B']
churn_dist.plot(kind='bar', ax=axes[0], color=colors, edgecolor='black')
axes[0].set_xlabel('Churn')
axes[0].set_ylabel('Count')
axes[0].set_title('Churn Distribution')
axes[0].set_xticklabels(['No Churn', 'Churn'], rotation=0)

axes[1].pie(churn_dist, labels=['No Churn', 'Churn'], autopct='%1.1f%%', 
            colors=colors, explode=[0, 0.1])
axes[1].set_title('Churn Percentage')

plt.tight_layout()
plt.show()

print("\nNote: This is an IMBALANCED dataset - we need to handle this!")

In [ ]:
# Churn rate by categorical features
categorical_cols = ['gender', 'SeniorCitizen', 'Partner', 'Dependents', 
                    'PhoneService', 'InternetService', 'Contract', 
                    'PaperlessBilling', 'PaymentMethod']

fig, axes = plt.subplots(3, 3, figsize=(16, 14))

for idx, col in enumerate(categorical_cols):
    ax = axes[idx // 3, idx % 3]
    
    churn_rate = df.groupby(col)['Churn'].mean().sort_values(ascending=False) * 100
    churn_rate.plot(kind='bar', ax=ax, color='teal', edgecolor='black')
    
    ax.set_xlabel(col)
    ax.set_ylabel('Churn Rate (%)')
    ax.set_title(f'Churn Rate by {col}')
    ax.tick_params(axis='x', rotation=45)
    ax.axhline(y=df['Churn'].mean() * 100, color='red', linestyle='--', label='Overall Rate')

plt.tight_layout()
plt.show()

In [ ]:
# Numerical features analysis
numerical_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

for idx, col in enumerate(numerical_cols):
    ax = axes[idx]
    
    # Distribution by churn status
    df[df['Churn'] == 0][col].hist(bins=30, alpha=0.6, label='No Churn', ax=ax, color='#4ECDC4')
    df[df['Churn'] == 1][col].hist(bins=30, alpha=0.6, label='Churn', ax=ax, color='#FF6B6B')
    
    ax.set_xlabel(col)
    ax.set_ylabel('Frequency')
    ax.set_title(f'{col} Distribution by Churn')
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
# Tenure analysis - key insight
plt.figure(figsize=(12, 5))

tenure_bins = [0, 6, 12, 24, 48, 72]
tenure_labels = ['0-6m', '6-12m', '12-24m', '24-48m', '48-72m']
df['tenure_group'] = pd.cut(df['tenure'], bins=tenure_bins, labels=tenure_labels)

churn_by_tenure = df.groupby('tenure_group')['Churn'].agg(['mean', 'count'])
churn_by_tenure['churn_rate'] = churn_by_tenure['mean'] * 100

plt.bar(churn_by_tenure.index, churn_by_tenure['churn_rate'], color='coral', edgecolor='black')
plt.xlabel('Tenure Group')
plt.ylabel('Churn Rate (%)')
plt.title('Churn Rate by Tenure Group\n(Shorter tenure = Higher churn!)')
plt.axhline(y=df['Churn'].mean() * 100, color='red', linestyle='--', label='Overall Rate')
plt.legend()

# Add counts on bars
for i, (rate, count) in enumerate(zip(churn_by_tenure['churn_rate'], churn_by_tenure['count'])):
    plt.text(i, rate + 1, f'n={count}', ha='center')

plt.show()

print("Key Insight: New customers (0-6 months) have MUCH higher churn rate!")

In [ ]:
# Contract type analysis - another key insight
plt.figure(figsize=(10, 6))

contract_analysis = df.groupby('Contract').agg({
    'Churn': ['mean', 'count'],
    'MonthlyCharges': 'mean',
    'tenure': 'mean'
}).round(2)

contract_analysis.columns = ['Churn_Rate', 'Count', 'Avg_Monthly', 'Avg_Tenure']
contract_analysis['Churn_Rate'] = contract_analysis['Churn_Rate'] * 100

print("Contract Analysis:")
print(contract_analysis)

# Visualize
contract_analysis['Churn_Rate'].plot(kind='bar', color=['#FF6B6B', '#FFEAA7', '#4ECDC4'], edgecolor='black')
plt.ylabel('Churn Rate (%)')
plt.title('Churn Rate by Contract Type\n(Month-to-month contracts have highest churn!)')
plt.xticks(rotation=0)
plt.show()

## Part 3: Data Preprocessing

In [ ]:
# Prepare features and target
# Drop customerID and tenure_group (created for analysis)
df_model = df.drop(['customerID', 'tenure_group'], axis=1)

X = df_model.drop('Churn', axis=1)
y = df_model['Churn']

print(f"Features shape: {X.shape}")
print(f"Target shape: {y.shape}")

In [ ]:
# Identify column types
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric features ({len(numeric_features)}): {numeric_features}")
print(f"\nCategorical features ({len(categorical_features)}): {categorical_features}")

In [ ]:
# Create preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

print("Preprocessing pipeline created!")

In [ ]:
# Split data with stratification (important for imbalanced data!)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"\nTraining class distribution:")
print(y_train.value_counts(normalize=True))

In [ ]:
# Transform data
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print(f"Processed training shape: {X_train_processed.shape}")
print(f"Processed test shape: {X_test_processed.shape}")

## Part 4: Handling Imbalanced Data

In [ ]:
# Method 1: Class Weights
# Many sklearn models support class_weight='balanced'

# Method 2: SMOTE (Synthetic Minority Over-sampling Technique)
# !pip install imbalanced-learn

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.under_sampling import RandomUnderSampler
    from imblearn.pipeline import Pipeline as ImbPipeline
    
    # Apply SMOTE
    smote = SMOTE(random_state=42)
    X_train_resampled, y_train_resampled = smote.fit_resample(X_train_processed, y_train)
    
    print("Before SMOTE:")
    print(y_train.value_counts())
    print(f"\nAfter SMOTE:")
    print(pd.Series(y_train_resampled).value_counts())
    
    use_smote = True
except ImportError:
    print("imbalanced-learn not installed. Using class weights instead.")
    X_train_resampled, y_train_resampled = X_train_processed, y_train
    use_smote = False

## Part 5: Model Building & Comparison

In [ ]:
# Evaluation function
def evaluate_classifier(model, X_train, X_test, y_train, y_test, model_name):
    """
    Train and evaluate a classification model
    """
    # Train
    model.fit(X_train, y_train)
    
    # Predict
    y_pred = model.predict(X_test)
    
    # Get probabilities if available
    if hasattr(model, 'predict_proba'):
        y_prob = model.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, y_prob)
    else:
        y_prob = None
        auc = None
    
    # Calculate metrics
    results = {
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred),
        'Recall': recall_score(y_test, y_pred),
        'F1 Score': f1_score(y_test, y_pred),
        'AUC': auc
    }
    
    return results, y_pred, y_prob

In [ ]:
# Define models
models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000),
    'Decision Tree': DecisionTreeClassifier(class_weight='balanced', max_depth=10, random_state=42),
    'Random Forest': RandomForestClassifier(class_weight='balanced', n_estimators=100, max_depth=15, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5)
}

In [ ]:
# Train and evaluate all models
results_list = []
predictions = {}
probabilities = {}

# Use resampled data if SMOTE was applied, otherwise original
X_tr = X_train_resampled if use_smote else X_train_processed
y_tr = y_train_resampled if use_smote else y_train

for name, model in models.items():
    print(f"Training {name}...")
    results, pred, prob = evaluate_classifier(model, X_tr, X_test_processed, y_tr, y_test, name)
    results_list.append(results)
    predictions[name] = pred
    probabilities[name] = prob

# Create comparison DataFrame
results_df = pd.DataFrame(results_list)
results_df = results_df.sort_values('F1 Score', ascending=False)

print("\n" + "=" * 80)
print("MODEL COMPARISON RESULTS")
print("=" * 80)
print(results_df.round(4).to_string(index=False))

In [ ]:
# Visualize comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Metrics comparison
metrics = ['Precision', 'Recall', 'F1 Score']
x = np.arange(len(results_df))
width = 0.25

for i, metric in enumerate(metrics):
    axes[0].bar(x + i*width, results_df[metric], width, label=metric)

axes[0].set_xticks(x + width)
axes[0].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[0].set_ylabel('Score')
axes[0].set_title('Model Comparison: Precision, Recall, F1')
axes[0].legend()
axes[0].set_ylim(0, 1)

# AUC comparison
auc_values = results_df['AUC'].fillna(0)
axes[1].bar(results_df['Model'], auc_values, color='teal')
axes[1].set_xticklabels(results_df['Model'], rotation=45, ha='right')
axes[1].set_ylabel('AUC Score')
axes[1].set_title('AUC-ROC Comparison')
axes[1].set_ylim(0.5, 1)

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(10, 8))

for name, prob in probabilities.items():
    if prob is not None:
        fpr, tpr, _ = roc_curve(y_test, prob)
        auc = roc_auc_score(y_test, prob)
        plt.plot(fpr, tpr, linewidth=2, label=f'{name} (AUC={auc:.3f})')

plt.plot([0, 1], [0, 1], 'k--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## Part 6: Hyperparameter Tuning

In [ ]:
# Tune the best performing model
# Using GridSearchCV with stratified k-fold

rf_param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 15, 20],
    'min_samples_split': [2, 5],
    'class_weight': ['balanced', 'balanced_subsample']
}

print("Starting GridSearchCV for Random Forest...")

rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42, n_jobs=-1),
    rf_param_grid,
    cv=StratifiedKFold(n_splits=5),
    scoring='f1',  # Optimize for F1 score
    n_jobs=-1,
    verbose=1
)

rf_grid.fit(X_train_processed, y_train)

In [ ]:
# Best parameters
print("\nBest Parameters:")
print(rf_grid.best_params_)
print(f"\nBest CV F1 Score: {rf_grid.best_score_:.4f}")

In [ ]:
# Evaluate best model
best_model = rf_grid.best_estimator_

best_results, best_pred, best_prob = evaluate_classifier(
    best_model, X_train_processed, X_test_processed, 
    y_train, y_test, 'Random Forest (Tuned)'
)

print("\n" + "=" * 50)
print("TUNED MODEL RESULTS")
print("=" * 50)
for key, value in best_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

In [ ]:
# Detailed classification report
print("\nClassification Report:")
print(classification_report(y_test, best_pred, target_names=['No Churn', 'Churn']))

In [ ]:
# Confusion Matrix
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, best_pred)

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Churn', 'Churn'],
            yticklabels=['No Churn', 'Churn'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix: Best Model')
plt.show()

# Interpret
tn, fp, fn, tp = cm.ravel()
print(f"\nConfusion Matrix Interpretation:")
print(f"True Negatives (correctly predicted No Churn): {tn}")
print(f"False Positives (incorrectly predicted Churn): {fp}")
print(f"False Negatives (MISSED CHURNERS!): {fn}")
print(f"True Positives (correctly predicted Churn): {tp}")
print(f"\nChurn Catch Rate: {tp/(tp+fn)*100:.1f}%")

## Part 7: Feature Importance & Business Insights

In [ ]:
# Feature importance
cat_features_encoded = preprocessor.named_transformers_['cat']['encoder'].get_feature_names_out(categorical_features)
all_features = numeric_features + list(cat_features_encoded)

importance_df = pd.DataFrame({
    'Feature': all_features,
    'Importance': best_model.feature_importances_
}).sort_values('Importance', ascending=False)

# Plot top 15
plt.figure(figsize=(12, 8))
top_features = importance_df.head(15)
colors = ['#FF6B6B' if 'Contract' in f or 'tenure' in f else 'teal' for f in top_features['Feature']]

plt.barh(range(len(top_features)), top_features['Importance'], color=colors)
plt.yticks(range(len(top_features)), top_features['Feature'])
plt.xlabel('Feature Importance')
plt.title('Top 15 Churn Predictors\n(Red = Actionable features)')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

In [ ]:
# Business insights visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# 1. Churn probability by contract type
contract_probs = X_test.copy()
contract_probs['Churn_Prob'] = best_prob
contract_probs['Actual_Churn'] = y_test.values

contract_probs.groupby('Contract')['Churn_Prob'].mean().sort_values(ascending=False).plot(
    kind='bar', ax=axes[0, 0], color=['#FF6B6B', '#FFEAA7', '#4ECDC4'], edgecolor='black'
)
axes[0, 0].set_ylabel('Average Churn Probability')
axes[0, 0].set_title('Churn Risk by Contract Type')
axes[0, 0].tick_params(axis='x', rotation=45)

# 2. High risk customers
contract_probs['Risk_Level'] = pd.cut(contract_probs['Churn_Prob'], 
                                       bins=[0, 0.3, 0.6, 1.0],
                                       labels=['Low', 'Medium', 'High'])
risk_counts = contract_probs['Risk_Level'].value_counts()
risk_counts.plot(kind='pie', ax=axes[0, 1], autopct='%1.1f%%', 
                 colors=['#4ECDC4', '#FFEAA7', '#FF6B6B'])
axes[0, 1].set_title('Customer Risk Distribution')

# 3. Tenure vs Churn Probability
axes[1, 0].scatter(contract_probs['tenure'], contract_probs['Churn_Prob'], 
                   alpha=0.3, c=contract_probs['Actual_Churn'], cmap='RdYlGn_r')
axes[1, 0].set_xlabel('Tenure (months)')
axes[1, 0].set_ylabel('Predicted Churn Probability')
axes[1, 0].set_title('Tenure vs Churn Risk')

# 4. Monthly charges vs Churn Probability
axes[1, 1].scatter(contract_probs['MonthlyCharges'], contract_probs['Churn_Prob'], 
                   alpha=0.3, c=contract_probs['Actual_Churn'], cmap='RdYlGn_r')
axes[1, 1].set_xlabel('Monthly Charges ($)')
axes[1, 1].set_ylabel('Predicted Churn Probability')
axes[1, 1].set_title('Monthly Charges vs Churn Risk')

plt.tight_layout()
plt.show()

## Part 8: Model Deployment Preparation

In [ ]:
# Create final pipeline
final_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', best_model)
])

# Retrain on full training data
final_pipeline.fit(X_train, y_train)

print("Final pipeline created and trained!")

In [ ]:
# Prediction function
def predict_churn(customer_data):
    """
    Predict churn probability for a customer
    
    Args:
        customer_data: dict with customer features
    
    Returns:
        dict with prediction and probability
    """
    input_df = pd.DataFrame([customer_data])
    prediction = final_pipeline.predict(input_df)[0]
    probability = final_pipeline.predict_proba(input_df)[0]
    
    return {
        'will_churn': bool(prediction),
        'churn_probability': round(probability[1], 3),
        'retention_priority': 'HIGH' if probability[1] > 0.6 else 'MEDIUM' if probability[1] > 0.3 else 'LOW'
    }

# Example prediction
sample_customer = {
    'gender': 'Female',
    'SeniorCitizen': 0,
    'Partner': 'No',
    'Dependents': 'No',
    'tenure': 3,
    'PhoneService': 'Yes',
    'InternetService': 'Fiber optic',
    'Contract': 'Month-to-month',
    'PaperlessBilling': 'Yes',
    'PaymentMethod': 'Electronic check',
    'MonthlyCharges': 85.50,
    'TotalCharges': 256.50
}

result = predict_churn(sample_customer)
print("Sample Customer Prediction:")
print(f"Will Churn: {result['will_churn']}")
print(f"Churn Probability: {result['churn_probability']*100:.1f}%")
print(f"Retention Priority: {result['retention_priority']}")

## Part 9: Conclusions & Business Recommendations

### Model Performance Summary

- **Best Model**: Random Forest (Tuned)
- **F1 Score**: ~0.XX (balance of precision and recall)
- **AUC-ROC**: ~0.XX (good discrimination ability)
- **Recall**: ~XX% (catches XX% of churners)

### Key Churn Predictors

1. **Contract Type**: Month-to-month contracts have highest churn risk
2. **Tenure**: New customers (< 12 months) are most at risk
3. **Monthly Charges**: Higher charges correlate with higher churn
4. **Payment Method**: Electronic check users churn more
5. **Internet Service**: Fiber optic customers have higher churn

### Business Recommendations

#### Immediate Actions (High Impact)

1. **Target Month-to-Month Customers**
   - Offer incentives to switch to longer contracts
   - Provide discounts for annual commitment

2. **New Customer Onboarding**
   - Implement 90-day check-in program
   - Offer loyalty rewards for first year

3. **Payment Method Migration**
   - Incentivize auto-pay enrollment
   - Reduce friction in payment process

#### Retention Strategy by Risk Level

| Risk Level | Probability | Action |
|------------|-------------|--------|
| **HIGH** | > 60% | Immediate outreach, premium offers |
| **MEDIUM** | 30-60% | Proactive engagement, satisfaction survey |
| **LOW** | < 30% | Standard service, occasional rewards |

### ROI Estimation

- Average customer lifetime value: $X,XXX
- Cost of retention offer: $XX-XXX
- If model catches 75% of churners and 30% accept retention offers:
  - Retained customers = XX% × high-risk customers
  - Revenue saved = Retained × LTV

---

### Next Steps

1. Deploy model as real-time API
2. Integrate with CRM system
3. A/B test retention strategies
4. Monitor model performance and retrain quarterly

In [ ]:
# Save model
import joblib

# Save the pipeline
# joblib.dump(final_pipeline, 'churn_prediction_model.pkl')
# print("Model saved!")

---

## Capstone Project Checklist

- [x] Problem Definition with Business Context
- [x] Data Collection and Understanding
- [x] Comprehensive EDA
- [x] Data Preprocessing Pipeline
- [x] Handling Imbalanced Data (SMOTE/Class Weights)
- [x] Multiple Model Comparison
- [x] Hyperparameter Tuning (GridSearchCV)
- [x] Model Evaluation with Multiple Metrics
- [x] Confusion Matrix Analysis
- [x] Feature Importance Analysis
- [x] Business Insights Generation
- [x] Final Pipeline for Deployment
- [x] Actionable Recommendations

**Congratulations! You've completed the Classification Capstone Project!**